# M4Trust Open-Source Model Benchmark

Bu notebook, M4Trust sisteminde yalnızca LLM'in üstlendiği structured extraction/reasoning görevini açık kaynak modellerle benchmark eder.

Kapsam:

- `contract_markdown + fixed_rag_context -> extraction_json`
- Ortak prompt, ortak JSON schema, ortak parser, ortak validator ve ortak evaluator
- BERTurk encoder baseline
- Qwen3-14B, DeepSeek-R1-Distill-Qwen-14B ve Llama 3.1 8B Instruct için opsiyonel generative runners

Kapsam dışı:

- PDF/DOCX/OCR parsing
- RAG retrieval veya embedding index oluşturma
- Payment state machine
- Moka adapter
- Video analizi
- İnsan onay akışı
- Production backend entegrasyonu

## Run Mode Warning

Default run sadece BERTurk/regex baseline'dır; bu bir gerçek açık kaynak LLM benchmark sonucu değildir.

Gerçek açık kaynak LLM benchmark için `RUN_MODELS["qwen3_14b"]`, `RUN_MODELS["deepseek_r1_qwen_14b"]` veya `RUN_MODELS["llama31_8b"]` değerlerinden en az birini `True` yapın. Llama 3.1 gated olabilir ve Hugging Face token isteyebilir.

14B modeller için `SMOKE_14B_SINGLE_SAMPLE=True` ilk çalıştırmada yalnızca ilk sample'ı koşturur. Smoke başarılıysa full dataset için bu flag'i `False` yapın.

## 1. Setup, Drive Mount, Runtime Check

Colab Pro H100 High-RAM hedeflenir. Notebook CPU'da da schema/parser/metrics smoke testlerini çalıştırabilir.

In [1]:
# Colab dependency setup. Re-run this cell after changing runtime.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q transformers accelerate bitsandbytes sentencepiece protobuf pydantic pandas scikit-learn tqdm rapidfuzz huggingface_hub tabulate
else:
    print("Not running in Colab. Skipping pip install cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.8 MB/s eta 0:00:00


In [1]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import re
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import pandas as pd
from pydantic import BaseModel, Field, ValidationError

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    from rapidfuzz import fuzz
except Exception:
    from difflib import SequenceMatcher

    class _FallbackFuzz:
        @staticmethod
        def token_set_ratio(left: Any, right: Any) -> float:
            return SequenceMatcher(None, str(left).lower(), str(right).lower()).ratio() * 100

        @staticmethod
        def partial_ratio(left: Any, right: Any) -> float:
            left_text = str(left).lower()
            right_text = str(right).lower()
            if not left_text or not right_text:
                return 0.0
            if left_text in right_text:
                return 100.0
            window = max(len(left_text), 1)
            best = 0.0
            for start in range(0, max(len(right_text) - window + 1, 1)):
                candidate = right_text[start : start + window]
                best = max(best, SequenceMatcher(None, left_text, candidate).ratio() * 100)
            return best

    fuzz = _FallbackFuzz()

try:
    import torch
except Exception:
    torch = None

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/m4trust_benchmark")
else:
    BASE_DIR = Path("./m4trust_benchmark")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
RAW_DIR = OUTPUT_DIR / "raw_predictions"
METRICS_DIR = OUTPUT_DIR / "metrics"
REPORTS_DIR = OUTPUT_DIR / "reports"

for path in [DATA_DIR, RAW_DIR, METRICS_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def get_gpu_info() -> dict[str, Any]:
    info = {
        "cuda_available": False,
        "gpu_name": None,
        "vram_gb": 0.0,
        "run_profile": "cpu_smoke",
    }
    if torch is None:
        return info
    info["cuda_available"] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["vram_gb"] = round(props.total_memory / 1024**3, 2)
        gpu_name_lower = str(info["gpu_name"]).lower()
        if "h100" in gpu_name_lower or info["vram_gb"] >= 70:
            info["run_profile"] = "h100_full"
        elif info["vram_gb"] >= 35:
            info["run_profile"] = "high_vram"
        else:
            info["run_profile"] = "low_vram"
    return info

GPU_INFO = get_gpu_info()
RUN_PROFILE = GPU_INFO["run_profile"]

print("CUDA available:", GPU_INFO["cuda_available"])
print("GPU:", GPU_INFO["gpu_name"])
print("VRAM GB:", GPU_INFO["vram_gb"])
print("RUN_PROFILE:", RUN_PROFILE)
print("BASE_DIR:", BASE_DIR)

Mounted at /content/drive
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM GB: 79.25
RUN_PROFILE: h100_full
BASE_DIR: /content/drive/MyDrive/m4trust_benchmark


## 2. Benchmark Configuration

Varsayılan ayar güvenli çalışır: BERTurk açık, büyük generative modeller kapalı. H100 runtime'da ilgili flag'i `True` yaparak tek tek çalıştırın.

In [2]:
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF_TOKEN başarıyla ayarlandı.")
except Exception as e:
    print("HF_TOKEN bulunamadı. Lütfen Colab Secrets paneline 'HF_TOKEN' adıyla eklediğinizden emin olun.")

HF_TOKEN başarıyla ayarlandı.


In [3]:
RUN_MODELS = {
    "berturk": True,
    "qwen3_14b": True,
    "deepseek_r1_qwen_14b": True,
    "llama31_8b": True,
}

# Set to a small integer for smoke tests; set to None for the full dataset.
MAX_SAMPLES = None

# 14B causal LMs are expensive even on H100. Keep this True for first-pass smoke runs.
SMOKE_14B_SINGLE_SAMPLE = False

# Optional Hugging Face token. Prefer Colab Secrets/env var over hardcoding.
HF_TOKEN = os.environ.get("HF_TOKEN") or None

GENERATION_CONFIG = {
    "do_sample": False,
    "max_new_tokens": 1400,
}

if RUN_PROFILE == "h100_full":
    DEFAULT_GENERATIVE_PRECISION = "bf16"
else:
    DEFAULT_GENERATIVE_PRECISION = "4bit"

print("DEFAULT_GENERATIVE_PRECISION:", DEFAULT_GENERATIVE_PRECISION)

DEFAULT_GENERATIVE_PRECISION: bf16


## 3. Extraction JSON Schema

In [4]:
Currency = Literal["TRY", "USD", "EUR", "OTHER"]
Trigger = Literal["approval", "e_invoice", "delivery_video", "manual_review"]


class Party(BaseModel):
    name: str | None = None
    tax_id: str | None = None


class GoodsItem(BaseModel):
    name: str | None = None
    quantity: float | None = None
    unit: str | None = None


class CommercialTerms(BaseModel):
    currency: Currency | None = None
    total_amount: float | None = None
    goods: list[GoodsItem] = Field(default_factory=list)
    delivery_deadline: str | None = None


class PaymentRule(BaseModel):
    milestone: str | None = None
    trigger: Trigger | None = None
    percentage: float | None = None
    required_evidence: list[str] = Field(default_factory=list)
    source_quote: str | None = None
    confidence: float | None = None


class Parties(BaseModel):
    buyer: Party = Field(default_factory=Party)
    seller: Party = Field(default_factory=Party)


class ExtractionJSON(BaseModel):
    contract_id: str | None = None
    parties: Parties = Field(default_factory=Parties)
    commercial_terms: CommercialTerms = Field(default_factory=CommercialTerms)
    payment_rules: list[PaymentRule] = Field(default_factory=list)
    risk_flags: list[str] = Field(default_factory=list)
    needs_manual_review: bool = False


COMPACT_SCHEMA_DESCRIPTION = {
    "contract_id": "string|null",
    "parties": {
        "buyer": {"name": "string|null", "tax_id": "string|null"},
        "seller": {"name": "string|null", "tax_id": "string|null"},
    },
    "commercial_terms": {
        "currency": "TRY|USD|EUR|OTHER|null",
        "total_amount": "number|null",
        "goods": [{"name": "string|null", "quantity": "number|null", "unit": "string|null"}],
        "delivery_deadline": "YYYY-MM-DD|null",
    },
    "payment_rules": [
        {
            "milestone": "string|null",
            "trigger": "approval|e_invoice|delivery_video|manual_review|null",
            "percentage": "number|null",
            "required_evidence": ["contract|e_irsaliye|video|approval|invoice|delivery"],
            "source_quote": "string|null",
            "confidence": "number|null",
        }
    ],
    "risk_flags": ["string"],
    "needs_manual_review": "boolean",
}

## 4. Dataset Loader + Embedded Toy Dataset

In [5]:
TOY_SAMPLES = [
    {
        "id": "sample_001",
        "contract_markdown": (
            "## Satış Sözleşmesi\n"
            "Alıcı: Atlas Gıda A.Ş. VKN: 1234567890. Satıcı: Bora Makine Ltd. VKN: 9876543210.\n"
            "Toplam bedel 100000 TRY'dir. Satıcı 10 adet endüstriyel mikser teslim edecektir.\n"
            "Teslim tarihi 2026-08-15 olarak belirlenmiştir. Ödeme, e-irsaliye onaylandıktan sonra %100 yapılacaktır."
        ),
        "rag_context": [
            {"source": "M4TRUST_POLICY", "madde_no": "P1", "text": "e-irsaliye onayı ödeme tetikleyicisi olarak kullanılabilir."}
        ],
        "gold": {
            "contract_id": "sample_001",
            "parties": {
                "buyer": {"name": "Atlas Gıda A.Ş.", "tax_id": "1234567890"},
                "seller": {"name": "Bora Makine Ltd.", "tax_id": "9876543210"},
            },
            "commercial_terms": {
                "currency": "TRY",
                "total_amount": 100000,
                "goods": [{"name": "endüstriyel mikser", "quantity": 10, "unit": "adet"}],
                "delivery_deadline": "2026-08-15",
            },
            "payment_rules": [
                {
                    "milestone": "Teslimat sonrası ödeme",
                    "trigger": "e_invoice",
                    "percentage": 100,
                    "required_evidence": ["e_irsaliye"],
                    "source_quote": "Ödeme, e-irsaliye onaylandıktan sonra %100 yapılacaktır.",
                    "confidence": 0.95,
                }
            ],
            "risk_flags": [],
            "needs_manual_review": False,
        },
    },
    {
        "id": "sample_002",
        "contract_markdown": (
            "## Tedarik Sözleşmesi\n"
            "Alıcı: Deniz Tekstil A.Ş. Satıcı: Ege Ambalaj Ltd.\n"
            "Sözleşme bedeli 25000 USD. Mallar: 5000 adet karton kutu.\n"
            "%40 sipariş onayı ile, %60 teslimat videosu ve irsaliye kontrolünden sonra ödenir."
        ),
        "rag_context": [
            {"source": "M4TRUST_POLICY", "madde_no": "P2", "text": "Teslimat videosu ikincil kanıt olarak değerlendirilebilir."}
        ],
        "gold": {
            "contract_id": "sample_002",
            "parties": {
                "buyer": {"name": "Deniz Tekstil A.Ş.", "tax_id": None},
                "seller": {"name": "Ege Ambalaj Ltd.", "tax_id": None},
            },
            "commercial_terms": {
                "currency": "USD",
                "total_amount": 25000,
                "goods": [{"name": "karton kutu", "quantity": 5000, "unit": "adet"}],
                "delivery_deadline": None,
            },
            "payment_rules": [
                {
                    "milestone": "Sipariş onayı",
                    "trigger": "approval",
                    "percentage": 40,
                    "required_evidence": ["approval"],
                    "source_quote": "%40 sipariş onayı ile",
                    "confidence": 0.9,
                },
                {
                    "milestone": "Teslimat sonrası ödeme",
                    "trigger": "delivery_video",
                    "percentage": 60,
                    "required_evidence": ["video", "e_irsaliye"],
                    "source_quote": "%60 teslimat videosu ve irsaliye kontrolünden sonra ödenir.",
                    "confidence": 0.9,
                },
            ],
            "risk_flags": [],
            "needs_manual_review": False,
        },
    },
    {
        "id": "sample_003",
        "contract_markdown": (
            "## Hizmet ve Mal Alım Sözleşmesi\n"
            "Alıcı: Kuzey Bilişim A.Ş. Satıcı: Nova Sistem Ltd.\n"
            "Toplam tutar 18000 EUR'dur. Teslim tarihi açıkça belirtilmemiştir.\n"
            "Ödeme şartı taraflarca ayrıca mutabık kalınacaktır. Bazı maddelerde peşinat geçse de oran yazılmamıştır."
        ),
        "rag_context": [
            {"source": "M4TRUST_POLICY", "madde_no": "P3", "text": "Belirsiz ödeme koşulları manuel inceleme gerektirir."}
        ],
        "gold": {
            "contract_id": "sample_003",
            "parties": {
                "buyer": {"name": "Kuzey Bilişim A.Ş.", "tax_id": None},
                "seller": {"name": "Nova Sistem Ltd.", "tax_id": None},
            },
            "commercial_terms": {
                "currency": "EUR",
                "total_amount": 18000,
                "goods": [],
                "delivery_deadline": None,
            },
            "payment_rules": [
                {
                    "milestone": "Belirsiz ödeme şartı",
                    "trigger": "manual_review",
                    "percentage": None,
                    "required_evidence": [],
                    "source_quote": "Ödeme şartı taraflarca ayrıca mutabık kalınacaktır.",
                    "confidence": 0.7,
                }
            ],
            "risk_flags": ["payment_terms_unclear", "delivery_deadline_missing"],
            "needs_manual_review": True,
        },
    },
]


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_no}: {exc}") from exc
    return rows


def keyword_text(value: Any) -> str:
    text = unicodedata.normalize("NFKD", str(value or "").casefold())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.replace("ı", "i")


def normalize_rag_context(rag_context: Any) -> list[dict[str, str | None]]:
    if rag_context is None:
        return []
    if isinstance(rag_context, str):
        return [{"source": "ai_studio_context", "madde_no": "1", "text": rag_context}]
    if not isinstance(rag_context, list):
        return [{"source": "ai_studio_context", "madde_no": "1", "text": str(rag_context)}]
    normalized = []
    for idx, item in enumerate(rag_context, start=1):
        if isinstance(item, dict):
            normalized.append({
                "source": str(item.get("source", "context")),
                "madde_no": str(item.get("madde_no", idx)),
                "text": str(item.get("text", "")),
            })
        else:
            normalized.append({
                "source": "ai_studio_context",
                "madde_no": str(idx),
                "text": str(item),
            })
    return normalized


def normalize_party(value: Any) -> dict[str, str | None]:
    if isinstance(value, dict):
        name = value.get("name")
        normalized_name = None if keyword_text(name) in {"", "belirsiz", "belirtilmemis", "yok"} else name
        return {
            "name": normalized_name,
            "tax_id": value.get("tax_id"),
        }
    if value is None:
        return {"name": None, "tax_id": None}
    name = str(value).strip()
    if keyword_text(name) in {"", "belirsiz", "belirtilmemis", "yok"}:
        name = None
    return {"name": name, "tax_id": None}


def parse_goods_string(value: str) -> dict[str, Any]:
    text = str(value).strip()
    match = re.match(r"^\s*(\d+(?:[.,]\d+)?)\s+([^\s]+)\s+(.+?)\s*$", text)
    if not match:
        return {"name": text, "quantity": None, "unit": None}
    quantity_text, unit, name = match.groups()
    try:
        quantity = float(quantity_text.replace(",", "."))
    except ValueError:
        return {"name": text, "quantity": None, "unit": None}
    return {"name": name.strip(), "quantity": quantity, "unit": unit.strip()}


def normalize_goods(value: Any) -> list[dict[str, Any]]:
    if value is None:
        return []
    items = value if isinstance(value, list) else [value]
    normalized = []
    for item in items:
        if isinstance(item, dict):
            normalized.append({
                "name": item.get("name"),
                "quantity": item.get("quantity"),
                "unit": item.get("unit"),
            })
        else:
            normalized.append(parse_goods_string(str(item)))
    return normalized


def normalize_required_evidence(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        merged: list[str] = []
        for item in value:
            for normalized in normalize_required_evidence(item):
                if normalized not in merged:
                    merged.append(normalized)
        return merged
    text = str(value).strip()
    if not text:
        return []
    lowered = keyword_text(text)
    evidence_rules = [
        ("contract", ["sozlesme", "imza", "imzali", "imzalanmis"]),
        ("e_irsaliye", ["irsaliye"]),
        ("invoice", ["fatura"]),
        ("video", ["video"]),
        ("delivery", ["teslim", "tutanak", "rapor", "kabul"]),
        ("approval", ["onay"]),
        ("payment_receipt", ["dekont", "banka"]),
        ("photo", ["fotograf", "fotograflı", "fotografli"]),
    ]
    matched = [
        label
        for label, tokens in evidence_rules
        if any(token in lowered for token in tokens)
    ]
    return matched if matched else [text]


def normalize_trigger(value: Any, rule: dict[str, Any] | None = None, gold: dict[str, Any] | None = None) -> str:
    pieces = [str(value or "")]
    if rule:
        pieces.extend(str(rule.get(key, "")) for key in ["milestone", "required_evidence", "source_quote"])
    if gold:
        pieces.extend(str(flag) for flag in gold.get("risk_flags", []) or [])
    text = keyword_text(" ".join(pieces))
    if any(token in text for token in ["belirsiz", "keyfi", "mutabakat", "celisk", "manual", "manuel", "inceleme"]):
        return "manual_review"
    if any(token in text for token in ["video", "portal"]):
        return "delivery_video"
    if any(token in text for token in ["irsaliye", "gib", "xml"]):
        return "e_invoice"
    approval_tokens = [
        "onay", "imza", "sozlesme", "siparis", "kabul", "teslim", "montaj",
        "kurulum", "aktif", "fatura", "rapor", "kickoff", "kalite", "hizmet",
        "yayina", "yayin", "baslama",
    ]
    if any(token in text for token in approval_tokens):
        return "approval"
    return "manual_review"


def normalize_total_amount(value: Any, contract_markdown: str, risk_flags: list[str]) -> float | None:
    unknown_text = keyword_text(" ".join([contract_markdown, *risk_flags]))
    unknown_tokens = ["unknown", "belirsiz", "belirtilmemis", "hesaplanacaktir", "sabit fiyat yok", "tutar belirtilmemis"]
    try:
        amount = float(value) if value is not None else None
    except (TypeError, ValueError):
        return None
    if amount == 0 and any(token in unknown_text for token in unknown_tokens):
        return None
    return amount


def normalize_optional_date(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    lowered = keyword_text(text)
    if lowered in {"belirtilmemis", "belirsiz", "none", "null", "yok"}:
        return None
    return text


def normalize_currency(value: Any) -> str | None:
    if value is None:
        return None
    currency = str(value).strip().upper()
    if currency == "TL":
        return "TRY"
    return currency if currency in {"TRY", "USD", "EUR", "OTHER"} else "OTHER"


def normalize_gold(gold: dict[str, Any], sample_id: str, contract_markdown: str) -> dict[str, Any]:
    raw_gold = copy.deepcopy(gold or {})
    risk_flags = raw_gold.get("risk_flags", []) or []
    if isinstance(risk_flags, str):
        risk_flags = [risk_flags]
    risk_flags = [str(flag) for flag in risk_flags]

    raw_terms = raw_gold.get("commercial_terms", {}) or {}
    commercial_terms = {
        "currency": normalize_currency(raw_terms.get("currency")),
        "total_amount": normalize_total_amount(raw_terms.get("total_amount"), contract_markdown, risk_flags),
        "goods": normalize_goods(raw_terms.get("goods", raw_gold.get("goods"))),
        "delivery_deadline": normalize_optional_date(raw_terms.get("delivery_deadline", raw_gold.get("delivery_deadline"))),
    }

    parties = raw_gold.get("parties", {}) or {}
    payment_rules = []
    for rule in raw_gold.get("payment_rules", []) or []:
        normalized_rule = copy.deepcopy(rule)
        normalized_rule["trigger"] = normalize_trigger(normalized_rule.get("trigger"), rule=normalized_rule, gold=raw_gold)
        normalized_rule["required_evidence"] = normalize_required_evidence(normalized_rule.get("required_evidence"))
        if normalized_rule.get("confidence") is not None:
            normalized_rule["confidence"] = float(normalized_rule["confidence"])
        if normalized_rule.get("percentage") is not None:
            normalized_rule["percentage"] = float(normalized_rule["percentage"])
        payment_rules.append(normalized_rule)

    return {
        "contract_id": raw_gold.get("contract_id") or sample_id,
        "parties": {
            "buyer": normalize_party(parties.get("buyer")),
            "seller": normalize_party(parties.get("seller")),
        },
        "commercial_terms": commercial_terms,
        "payment_rules": payment_rules,
        "risk_flags": risk_flags,
        "needs_manual_review": bool(raw_gold.get("needs_manual_review", False)),
    }


def normalize_sample(sample: dict[str, Any]) -> dict[str, Any]:
    normalized = copy.deepcopy(sample)
    sample_id = str(normalized.get("id") or normalized.get("gold", {}).get("contract_id") or "sample")
    contract_markdown = str(normalized.get("contract_markdown", ""))
    normalized["id"] = sample_id
    normalized["contract_markdown"] = contract_markdown
    normalized["rag_context"] = normalize_rag_context(normalized.get("rag_context", []))
    normalized["gold"] = normalize_gold(normalized.get("gold", {}), sample_id, contract_markdown)
    return normalized


def normalize_samples(samples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [normalize_sample(sample) for sample in samples]


def validate_gold_samples(samples: list[dict[str, Any]]) -> None:
    for sample in samples:
        ExtractionJSON.model_validate(sample["gold"])
    print(f"Gold schema validation passed for {len(samples)} samples.")


def load_samples(data_path: str | Path) -> list[dict[str, Any]]:
    path = Path(data_path)
    if path.exists():
        samples = load_jsonl(path)
        print(f"Loaded {len(samples)} samples from {path}")
    else:
        samples = TOY_SAMPLES
        print(f"{path} not found. Using {len(samples)} embedded toy samples.")
    samples = normalize_samples(samples)
    if MAX_SAMPLES is not None:
        samples = samples[:MAX_SAMPLES]
    validate_gold_samples(samples)
    return samples


SAMPLES = load_samples(DATA_DIR / "benchmark_samples.jsonl")

Loaded 20 samples from /content/drive/MyDrive/m4trust_benchmark/data/benchmark_samples.jsonl
Gold schema validation passed for 20 samples.


## 4.1 Dataset Report

In [6]:
def build_dataset_report(samples: list[dict[str, Any]]) -> dict[str, Any]:
    triggers = []
    rule_counts = []
    manual_review_count = 0
    risk_sample_count = 0
    for sample in samples:
        gold = sample["gold"]
        if gold.get("needs_manual_review"):
            manual_review_count += 1
        if gold.get("risk_flags"):
            risk_sample_count += 1
        rules = gold.get("payment_rules", []) or []
        rule_counts.append(len(rules))
        triggers.extend(rule.get("trigger") for rule in rules if isinstance(rule, dict))
    return {
        "dataset_size": len(samples),
        "manual_review_count": manual_review_count,
        "risk_sample_count": risk_sample_count,
        "model_run_mode_hint": (
            "open-source LLM benchmark run"
            if any(RUN_MODELS.get(name, False) and MODEL_REGISTRY.get(name, {}).get("kind") == "causal_lm" for name in RUN_MODELS)
            else "baseline/smoke only"
        ) if "MODEL_REGISTRY" in globals() else (
            "open-source LLM benchmark run"
            if any(RUN_MODELS.get(name, False) for name in ["qwen3_14b", "deepseek_r1_qwen_14b", "llama31_8b"])
            else "baseline/smoke only"
        ),
        "trigger_distribution": pd.Series(triggers).value_counts().to_dict() if triggers else {},
        "payment_rule_count_distribution": pd.Series(rule_counts).value_counts().sort_index().to_dict() if rule_counts else {},
    }


DATASET_REPORT = build_dataset_report(SAMPLES)
display(pd.DataFrame([{
    "dataset_size": DATASET_REPORT["dataset_size"],
    "manual_review_count": DATASET_REPORT["manual_review_count"],
    "risk_sample_count": DATASET_REPORT["risk_sample_count"],
    "model_run_mode_hint": DATASET_REPORT["model_run_mode_hint"],
}]))
display(pd.DataFrame(DATASET_REPORT["trigger_distribution"].items(), columns=["trigger", "count"]))
display(pd.DataFrame(DATASET_REPORT["payment_rule_count_distribution"].items(), columns=["payment_rule_count", "sample_count"]))

,dataset_size,manual_review_count,risk_sample_count,model_run_mode_hint
0,20,7,10,open-source LLM benchmark run


,trigger,count
0,approval,20
1,manual_review,6
2,e_invoice,1
3,delivery_video,1


,payment_rule_count,sample_count
0,1,14
1,2,4
2,3,2


## 5. Prompt Builder

In [7]:
SYSTEM_PROMPT = """Sen M4Trust sözleşme extraction modelisin.

Görevin:
Verilen sözleşme markdown'u ve RAG context'i üzerinden tek bir extraction_json üretmek.

Kurallar:
- Sadece verilen sözleşme markdown'u ve RAG context'i kullan.
- JSON dışında hiçbir açıklama yazma.
- Para hareketi kararı verme.
- Ödeme aksiyonu yürütme.
- Emin olmadığın alanlarda null kullan.
- Belirsizlik varsa risk_flags ekle ve needs_manual_review=true yap.
- source_quote alanı mümkün olduğunca sözleşme metninden birebir kısa bir alıntı olmalı.
- payment_rules yüzdeleri mümkünse toplam 100 olmalı.
- trigger yalnızca şu değerlerden biri olmalı: approval, e_invoice, delivery_video, manual_review.
"""


def build_prompt(sample: dict[str, Any]) -> list[dict[str, str]]:
    rag_context = json.dumps(sample.get("rag_context", []), ensure_ascii=False, indent=2)
    schema_text = json.dumps(COMPACT_SCHEMA_DESCRIPTION, ensure_ascii=False, indent=2)
    user_prompt = f"""[EXTRACTION_JSON_SCHEMA]
{schema_text}

[RAG_CONTEXT]
{rag_context}

[CONTRACT_MARKDOWN]
{sample["contract_markdown"]}

Return only valid JSON."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


print(build_prompt(SAMPLES[0])[1]["content"][:1000])

[EXTRACTION_JSON_SCHEMA]
{
  "contract_id": "string|null",
  "parties": {
    "buyer": {
      "name": "string|null",
      "tax_id": "string|null"
    },
    "seller": {
      "name": "string|null",
      "tax_id": "string|null"
    }
  },
  "commercial_terms": {
    "currency": "TRY|USD|EUR|OTHER|null",
    "total_amount": "number|null",
    "goods": [
      {
        "name": "string|null",
        "quantity": "number|null",
        "unit": "string|null"
      }
    ],
    "delivery_deadline": "YYYY-MM-DD|null"
  },
  "payment_rules": [
    {
      "milestone": "string|null",
      "trigger": "approval|e_invoice|delivery_video|manual_review|null",
      "percentage": "number|null",
      "required_evidence": [
        "contract|e_irsaliye|video|approval|invoice|delivery"
      ],
      "source_quote": "string|null",
      "confidence": "number|null"
    }
  ],
  "risk_flags": [
    "string"
  ],
  "needs_manual_review": "boolean"
}

[RAG_CONTEXT]
[
  {
    "source": "ai_studio_contex

## 6. JSON Parser and Light Repair

In [8]:
TRAILING_COMMA_RE = re.compile(r",\s*([}\]])")


def strip_think_blocks(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)


def extract_json_candidate(raw_output: str) -> str | None:
    if not raw_output:
        return None
    text = strip_think_blocks(raw_output).strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        return fenced.group(1).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start : end + 1].strip()
    return text if text.startswith("{") else None


def parse_model_json(raw_output: str) -> dict[str, Any]:
    candidate = extract_json_candidate(raw_output)
    if candidate is None:
        return {
            "parse_success": False,
            "parsed_json": None,
            "parse_error": "No JSON object candidate found",
        }
    attempts = [candidate, TRAILING_COMMA_RE.sub(r"\1", candidate)]
    last_error = None
    for attempt in attempts:
        try:
            return {
                "parse_success": True,
                "parsed_json": json.loads(attempt),
                "parse_error": None,
            }
        except json.JSONDecodeError as exc:
            last_error = str(exc)
    return {
        "parse_success": False,
        "parsed_json": None,
        "parse_error": last_error,
    }


PARSER_TESTS = [
    '{"contract_id": "x", "payment_rules": []}',
    '```json\n{"contract_id": "x", "payment_rules": []}\n```',
    'Açıklama\n{"contract_id": "x", "payment_rules": []}\nBitti',
    'JSON yok burada',
]

for raw in PARSER_TESTS:
    print(parse_model_json(raw))

{'parse_success': True, 'parsed_json': {'contract_id': 'x', 'payment_rules': []}, 'parse_error': None}
{'parse_success': True, 'parsed_json': {'contract_id': 'x', 'payment_rules': []}, 'parse_error': None}
{'parse_success': True, 'parsed_json': {'contract_id': 'x', 'payment_rules': []}, 'parse_error': None}
{'parse_success': False, 'parsed_json': None, 'parse_error': 'No JSON object candidate found'}


## 7. Mini Validator

In [9]:
def flatten_context_text(sample: dict[str, Any]) -> str:
    rag_parts = []
    for item in sample.get("rag_context", []) or []:
        if isinstance(item, dict):
            rag_parts.append(str(item.get("text", "")))
        else:
            rag_parts.append(str(item))
    rag_text = "\n".join(rag_parts)
    return f"{sample.get('contract_markdown', '')}\n{rag_text}"


def grounding_sentences(sample: dict[str, Any]) -> list[str]:
    context_parts = [sample.get("contract_markdown", "")]
    for item in sample.get("rag_context", []) or []:
        if isinstance(item, dict):
            context_parts.append(str(item.get("text", "")))
        else:
            context_parts.append(str(item))
    sentences: list[str] = []
    for part in context_parts:
        sentences.extend(split_sentences(part) if "split_sentences" in globals() else re.split(r"(?<=[.!?])\s+|\n+", part))
    return [sentence.strip() for sentence in sentences if sentence and sentence.strip()]


def strip_markdown_symbols(value: str | None) -> str:
    if not value:
        return ""
    text = str(value)
    text = re.sub(r"^[#>*\-\s]+", "", text)
    text = re.sub(r"[*_`~\[\]()>#]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def meaningful_quote_text(value: str | None) -> str:
    return strip_markdown_symbols(value).strip()


def quote_has_business_content(value: str | None) -> bool:
    text = keyword_text(meaningful_quote_text(value))
    content_tokens = [
        "odeme", "oden", "ode", "tahsil", "teslim", "sevk", "fatura",
        "irsaliye", "onay", "imza", "kabul", "rapor", "video", "portal",
        "tutar", "bedel", "kurulum", "montaj", "hizmet", "aktif", "basla",
        "hak edis", "hakedis", "dekont",
    ]
    return any(token in text for token in content_tokens)


def quote_looks_like_heading_only(value: str | None) -> bool:
    raw = str(value or "").strip()
    cleaned = meaningful_quote_text(raw)
    if not cleaned:
        return True
    if raw.lstrip().startswith("#"):
        return True
    if cleaned.endswith(":"):
        return True
    if not quote_has_business_content(cleaned):
        return True
    return False


def quote_is_usable(value: str | None) -> bool:
    cleaned = meaningful_quote_text(value)
    if len(cleaned) < 15:
        return False
    if quote_looks_like_heading_only(value):
        return False
    return True


def quote_is_grounded(quote: str | None, sample: dict[str, Any], threshold: int = 85) -> bool:
    if not quote or not quote_is_usable(quote):
        return False
    cleaned_quote = meaningful_quote_text(quote)
    normalized_quote = normalize_text(cleaned_quote) if "normalize_text" in globals() else re.sub(r"\s+", " ", cleaned_quote.strip().lower())
    if not normalized_quote:
        return False
    for sentence in grounding_sentences(sample):
        cleaned_sentence = meaningful_quote_text(sentence)
        normalized_sentence = normalize_text(cleaned_sentence) if "normalize_text" in globals() else re.sub(r"\s+", " ", cleaned_sentence.strip().lower())
        if normalized_quote in normalized_sentence:
            return True
        if fuzz.partial_ratio(normalized_quote, normalized_sentence) >= threshold:
            return True
    # Last-resort exact fallback for context fragments that do not split cleanly. No fuzzy haystack matching.
    haystack = normalize_text(meaningful_quote_text(flatten_context_text(sample))) if "normalize_text" in globals() else meaningful_quote_text(flatten_context_text(sample)).lower()
    if normalized_quote in haystack:
        return True
    return False


def validate_prediction(prediction: dict[str, Any] | None, sample: dict[str, Any]) -> dict[str, Any]:
    errors = []
    if prediction is None:
        return {
            "schema_valid": False,
            "validator_pass": False,
            "validator_errors": ["prediction_missing"],
        }
    try:
        parsed = ExtractionJSON.model_validate(prediction)
        schema_valid = True
    except ValidationError as exc:
        return {
            "schema_valid": False,
            "validator_pass": False,
            "validator_errors": [f"schema_error: {exc.errors()}"],
        }

    if not parsed.payment_rules:
        errors.append("payment_rules_empty")

    percentages = []
    for idx, rule in enumerate(parsed.payment_rules):
        if rule.percentage is None:
            errors.append(f"payment_rules[{idx}].percentage_missing")
        else:
            if not 0 <= rule.percentage <= 100:
                errors.append(f"payment_rules[{idx}].percentage_out_of_range")
            percentages.append(rule.percentage)
        if rule.trigger is None:
            errors.append(f"payment_rules[{idx}].trigger_missing")
        if not rule.required_evidence:
            errors.append(f"payment_rules[{idx}].required_evidence_empty")
        if rule.confidence is None:
            errors.append(f"payment_rules[{idx}].confidence_missing")
        elif not 0 <= rule.confidence <= 1:
            errors.append(f"payment_rules[{idx}].confidence_out_of_range")
        if not rule.source_quote:
            errors.append(f"payment_rules[{idx}].source_quote_empty")
        elif not quote_is_usable(rule.source_quote):
            errors.append(f"payment_rules[{idx}].source_quote_heading_or_too_short")
        elif not quote_is_grounded(rule.source_quote, sample):
            errors.append(f"payment_rules[{idx}].source_quote_not_grounded")

    if percentages and abs(sum(percentages) - 100) > 1:
        errors.append(f"payment_percentage_sum_not_100: {sum(percentages)}")

    if not isinstance(parsed.needs_manual_review, bool):
        errors.append("needs_manual_review_not_boolean")

    gold = sample.get("gold", {})
    gold_amount = get_path(gold, ["commercial_terms", "total_amount"]) if "get_path" in globals() else gold.get("commercial_terms", {}).get("total_amount")
    gold_deadline = get_path(gold, ["commercial_terms", "delivery_deadline"]) if "get_path" in globals() else gold.get("commercial_terms", {}).get("delivery_deadline")
    gold_buyer = get_path(gold, ["parties", "buyer", "name"]) if "get_path" in globals() else gold.get("parties", {}).get("buyer", {}).get("name")
    gold_seller = get_path(gold, ["parties", "seller", "name"]) if "get_path" in globals() else gold.get("parties", {}).get("seller", {}).get("name")
    if parsed.commercial_terms.total_amount is None and gold_amount is not None:
        errors.append("commercial_terms.total_amount_missing")
    if parsed.commercial_terms.delivery_deadline is None and gold_deadline is not None:
        errors.append("commercial_terms.delivery_deadline_missing")
    if not parsed.parties.buyer.name and not parsed.parties.seller.name and (gold_buyer or gold_seller):
        errors.append("parties.buyer_and_seller_missing")

    return {
        "schema_valid": schema_valid,
        "validator_pass": len(errors) == 0,
        "validator_errors": errors,
    }


print(validate_prediction(SAMPLES[0]["gold"], SAMPLES[0]))
bad_prediction = json.loads(json.dumps(SAMPLES[0]["gold"], ensure_ascii=False))
bad_prediction["payment_rules"][0]["percentage"] = 40
print(validate_prediction(bad_prediction, SAMPLES[0]))

{'schema_valid': True, 'validator_pass': True, 'validator_errors': []}
{'schema_valid': True, 'validator_pass': False, 'validator_errors': ['payment_percentage_sum_not_100: 40.0']}


## 8. Metrics

In [10]:
def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).strip().lower())


def fuzzy_score(pred: Any, gold: Any) -> float:
    pred_text = normalize_text(pred)
    gold_text = normalize_text(gold)
    if not pred_text and not gold_text:
        return 1.0
    if not pred_text or not gold_text:
        return 0.0
    return fuzz.token_set_ratio(pred_text, gold_text) / 100


def exact_score(pred: Any, gold: Any) -> float:
    return float(normalize_text(pred) == normalize_text(gold))


def numeric_score(pred: Any, gold: Any, tolerance: float = 1.0) -> float:
    if pred is None and gold is None:
        return 1.0
    if pred is None or gold is None:
        return 0.0
    try:
        return float(abs(float(pred) - float(gold)) <= tolerance)
    except Exception:
        return 0.0


def list_f1(pred_items: list[Any], gold_items: list[Any]) -> float:
    pred_set = {normalize_text(item) for item in pred_items if normalize_text(item)}
    gold_set = {normalize_text(item) for item in gold_items if normalize_text(item)}
    if not pred_set and not gold_set:
        return 1.0
    if not pred_set or not gold_set:
        return 0.0
    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set) if gold_set else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def get_path(data: dict[str, Any], path: list[Any], default: Any = None) -> Any:
    current: Any = data
    for part in path:
        if isinstance(part, int):
            if not isinstance(current, list) or len(current) <= part:
                return default
            current = current[part]
        elif isinstance(current, dict):
            current = current.get(part, default)
        else:
            return default
    return current


def score_prediction(prediction: dict[str, Any] | None, gold: dict[str, Any], sample: dict[str, Any]) -> dict[str, float]:
    if prediction is None:
        metric_names = [
            "party_name_fuzzy_f1", "tax_id_exact_accuracy", "currency_accuracy",
            "total_amount_accuracy", "delivery_deadline_accuracy", "goods_name_fuzzy_f1",
            "goods_quantity_accuracy", "payment_rule_count_accuracy", "payment_percentage_accuracy",
            "trigger_accuracy", "required_evidence_f1", "risk_flags_f1",
            "needs_manual_review_accuracy", "source_quote_grounding_rate", "critical_field_pass_rate", "field_f1",
        ]
        return {name: 0.0 for name in metric_names}

    buyer_name = fuzzy_score(get_path(prediction, ["parties", "buyer", "name"]), get_path(gold, ["parties", "buyer", "name"]))
    seller_name = fuzzy_score(get_path(prediction, ["parties", "seller", "name"]), get_path(gold, ["parties", "seller", "name"]))
    buyer_tax = exact_score(get_path(prediction, ["parties", "buyer", "tax_id"]), get_path(gold, ["parties", "buyer", "tax_id"]))
    seller_tax = exact_score(get_path(prediction, ["parties", "seller", "tax_id"]), get_path(gold, ["parties", "seller", "tax_id"]))

    pred_goods = get_path(prediction, ["commercial_terms", "goods"], []) or []
    gold_goods = get_path(gold, ["commercial_terms", "goods"], []) or []
    pred_good_names = [item.get("name") for item in pred_goods if isinstance(item, dict)]
    gold_good_names = [item.get("name") for item in gold_goods if isinstance(item, dict)]
    pred_good_qty = pred_goods[0].get("quantity") if pred_goods and isinstance(pred_goods[0], dict) else None
    gold_good_qty = gold_goods[0].get("quantity") if gold_goods and isinstance(gold_goods[0], dict) else None

    pred_rules = prediction.get("payment_rules", []) or []
    gold_rules = gold.get("payment_rules", []) or []
    pred_percentages = [rule.get("percentage") for rule in pred_rules if isinstance(rule, dict) and rule.get("percentage") is not None]
    gold_percentages = [rule.get("percentage") for rule in gold_rules if isinstance(rule, dict) and rule.get("percentage") is not None]
    pred_triggers = [rule.get("trigger") for rule in pred_rules if isinstance(rule, dict)]
    gold_triggers = [rule.get("trigger") for rule in gold_rules if isinstance(rule, dict)]
    pred_evidence = []
    gold_evidence = []
    for rule in pred_rules:
        if isinstance(rule, dict):
            pred_evidence.extend(rule.get("required_evidence", []) or [])
    for rule in gold_rules:
        if isinstance(rule, dict):
            gold_evidence.extend(rule.get("required_evidence", []) or [])

    quote_grounding_values = []
    for rule in pred_rules:
        if isinstance(rule, dict):
            quote_grounding_values.append(float(quote_is_grounded(rule.get("source_quote"), sample)))

    payment_percentage_accuracy = 1.0 if pred_percentages == gold_percentages else 0.0
    if len(pred_percentages) == len(gold_percentages) and pred_percentages:
        payment_percentage_accuracy = sum(
            numeric_score(pred, gold_value, tolerance=0.5)
            for pred, gold_value in zip(pred_percentages, gold_percentages)
        ) / len(gold_percentages)

    metrics = {
        "party_name_fuzzy_f1": (buyer_name + seller_name) / 2,
        "tax_id_exact_accuracy": (buyer_tax + seller_tax) / 2,
        "currency_accuracy": exact_score(
            get_path(prediction, ["commercial_terms", "currency"]),
            get_path(gold, ["commercial_terms", "currency"]),
        ),
        "total_amount_accuracy": numeric_score(
            get_path(prediction, ["commercial_terms", "total_amount"]),
            get_path(gold, ["commercial_terms", "total_amount"]),
        ),
        "delivery_deadline_accuracy": exact_score(
            get_path(prediction, ["commercial_terms", "delivery_deadline"]),
            get_path(gold, ["commercial_terms", "delivery_deadline"]),
        ),
        "goods_name_fuzzy_f1": list_f1(pred_good_names, gold_good_names),
        "goods_quantity_accuracy": numeric_score(pred_good_qty, gold_good_qty),
        "payment_rule_count_accuracy": float(len(pred_rules) == len(gold_rules)),
        "payment_percentage_accuracy": payment_percentage_accuracy,
        "trigger_accuracy": list_f1(pred_triggers, gold_triggers),
        "required_evidence_f1": list_f1(pred_evidence, gold_evidence),
        "risk_flags_f1": list_f1(prediction.get("risk_flags", []) or [], gold.get("risk_flags", []) or []),
        "needs_manual_review_accuracy": exact_score(prediction.get("needs_manual_review"), gold.get("needs_manual_review")),
        "source_quote_grounding_rate": sum(quote_grounding_values) / len(quote_grounding_values) if quote_grounding_values else 0.0,
    }
    critical_values = [
        float(buyer_name >= 0.85),
        float(seller_name >= 0.85),
        metrics["currency_accuracy"],
        metrics["total_amount_accuracy"],
        metrics["delivery_deadline_accuracy"],
        metrics["payment_rule_count_accuracy"],
        metrics["payment_percentage_accuracy"],
        metrics["trigger_accuracy"],
    ]
    metrics["critical_field_pass_rate"] = sum(critical_values) / len(critical_values)
    field_keys = [
        "party_name_fuzzy_f1", "tax_id_exact_accuracy", "currency_accuracy",
        "total_amount_accuracy", "delivery_deadline_accuracy", "goods_name_fuzzy_f1",
        "goods_quantity_accuracy", "payment_rule_count_accuracy", "payment_percentage_accuracy",
        "trigger_accuracy", "required_evidence_f1", "risk_flags_f1",
        "needs_manual_review_accuracy", "source_quote_grounding_rate",
    ]
    metrics["field_f1"] = sum(metrics[key] for key in field_keys) / len(field_keys)
    return metrics


gold_scores = [score_prediction(sample["gold"], sample["gold"], sample) for sample in SAMPLES]
GOLD_VS_GOLD_DF = pd.DataFrame(gold_scores)
GOLD_VS_GOLD_SUMMARY = GOLD_VS_GOLD_DF.mean(numeric_only=True).sort_values(ascending=True)
display(GOLD_VS_GOLD_DF)
display(GOLD_VS_GOLD_SUMMARY.to_frame("gold_vs_gold_average"))
assert float(GOLD_VS_GOLD_DF["field_f1"].mean()) > 0.95, "Gold-vs-gold average field_f1 must be > 0.95"

,party_name_fuzzy_f1,tax_id_exact_accuracy,currency_accuracy,total_amount_accuracy,delivery_deadline_accuracy,goods_name_fuzzy_f1,goods_quantity_accuracy,payment_rule_count_accuracy,payment_percentage_accuracy,trigger_accuracy,required_evidence_f1,risk_flags_f1,needs_manual_review_accuracy,source_quote_grounding_rate,critical_field_pass_rate,field_f1
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


,gold_vs_gold_average
party_name_fuzzy_f1,1.0
tax_id_exact_accuracy,1.0
currency_accuracy,1.0
total_amount_accuracy,1.0
delivery_deadline_accuracy,1.0
goods_name_fuzzy_f1,1.0
goods_quantity_accuracy,1.0
payment_rule_count_accuracy,1.0
payment_percentage_accuracy,1.0
trigger_accuracy,1.0


## 9. Model Registry

In [11]:
MODEL_REGISTRY = {
    "berturk": {
        "kind": "encoder_baseline",
        "model_id": "dbmdz/bert-base-turkish-cased",
    },
    "qwen3_14b": {
        "kind": "causal_lm",
        "model_id": "Qwen/Qwen3-14B",
        "default_precision": "bf16",
        "fallback_precision": "4bit",
        "requires_hf_token": False,
    },
    "deepseek_r1_qwen_14b": {
        "kind": "causal_lm",
        "model_id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
        "default_precision": "bf16",
        "fallback_precision": "4bit",
        "requires_hf_token": False,
    },
    "llama31_8b": {
        "kind": "causal_lm",
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "default_precision": "bf16",
        "fallback_precision": "4bit",
        "requires_hf_token": True,
    },
}

## 10. BERTurk Encoder Baseline

In [12]:
class BaseRunner:
    model_name: str
    model_id: str
    runner_mode: str = "unknown"
    runner_notes: list[str] = []

    def run(self, sample: dict[str, Any], prompt_messages: list[dict[str, str]]) -> str:
        raise NotImplementedError

    def unload(self) -> None:
        pass


def split_sentences(text: str) -> list[str]:
    chunks = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [chunk.strip() for chunk in chunks if chunk.strip()]


def regex_candidates(text: str) -> dict[str, list[str]]:
    return {
        "dates": re.findall(r"\b\d{4}-\d{2}-\d{2}\b|\b\d{2}[./]\d{2}[./]\d{4}\b", text),
        "amounts": re.findall(r"\b\d[\d.,]*\s*(?:TRY|TL|USD|EUR)\b", text, flags=re.IGNORECASE),
        "percentages": re.findall(r"%\s*\d+(?:[.,]\d+)?|yüzde\s+\d+(?:[.,]\d+)?", text, flags=re.IGNORECASE),
        "tax_ids": re.findall(r"\b\d{10,11}\b", text),
    }


def parse_amount(amount_text: str | None) -> tuple[float | None, str | None]:
    if not amount_text:
        return None, None
    currency_match = re.search(r"(TRY|TL|USD|EUR)", amount_text, flags=re.IGNORECASE)
    currency_raw = currency_match.group(1).upper() if currency_match else None
    currency = "TRY" if currency_raw == "TL" else currency_raw
    number_match = re.search(r"\d[\d.,]*", amount_text)
    if not number_match:
        return None, currency
    number_text = number_match.group(0).replace(".", "").replace(",", ".")
    try:
        return float(number_text), currency
    except ValueError:
        return None, currency


def parse_percentage(text: str) -> float | None:
    match = re.search(r"\d+(?:[.,]\d+)?", text)
    if not match:
        return None
    return float(match.group(0).replace(",", "."))


def detect_trigger(sentence: str) -> str:
    lower = sentence.lower()
    if "irsaliye" in lower or "e-irsaliye" in lower:
        return "e_invoice"
    if "video" in lower:
        return "delivery_video"
    if "onay" in lower:
        return "approval"
    return "manual_review"


def detect_evidence(sentence: str) -> list[str]:
    lower = sentence.lower()
    evidence = []
    if "irsaliye" in lower:
        evidence.append("e_irsaliye")
    if "video" in lower:
        evidence.append("video")
    if "onay" in lower:
        evidence.append("approval")
    if "teslim" in lower:
        evidence.append("delivery")
    return evidence


class BerturkBaselineRunner(BaseRunner):
    def __init__(self, model_name: str, model_id: str):
        self.model_name = model_name
        self.model_id = model_id
        self.tokenizer = None
        self.model = None
        self.device = "cpu"
        self.runner_mode = "berturk_encoder"
        self.runner_notes: list[str] = []
        self._load_encoder()

    def _load_encoder(self) -> None:
        try:
            from transformers import AutoModel, AutoTokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
            self.model = AutoModel.from_pretrained(self.model_id)
            if torch is not None and torch.cuda.is_available():
                self.device = "cuda"
                self.model.to(self.device)
            self.model.eval()
            print(f"Loaded BERTurk encoder on {self.device}")
        except Exception as exc:
            note = f"BERTurk yüklenemedi; regex-only fallback kullanıldı. Error: {exc}"
            print(note)
            self.tokenizer = None
            self.model = None
            self.runner_mode = "regex_only_fallback"
            self.runner_notes.append("BERTurk weights not loaded; regex-only fallback used")
            self.runner_notes.append(note)

    def _sentence_embeddings(self, sentences: list[str]):
        if self.tokenizer is None or self.model is None or torch is None:
            return None
        encoded = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt", max_length=256)
        encoded = {key: value.to(self.device) for key, value in encoded.items()}
        with torch.no_grad():
            output = self.model(**encoded)
            mask = encoded["attention_mask"].unsqueeze(-1)
            embeddings = (output.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return embeddings

    def _best_sentence(self, query: str, sentences: list[str], fallback_keywords: list[str]) -> str | None:
        if not sentences:
            return None
        if self.tokenizer is not None and self.model is not None and torch is not None:
            embeddings = self._sentence_embeddings([query] + sentences)
            if embeddings is not None:
                query_emb = embeddings[0:1]
                sent_emb = embeddings[1:]
                sims = torch.nn.functional.cosine_similarity(query_emb, sent_emb)
                return sentences[int(torch.argmax(sims).item())]
        for sentence in sentences:
            lower = sentence.lower()
            if any(keyword in lower for keyword in fallback_keywords):
                return sentence
        return sentences[0]

    def run(self, sample: dict[str, Any], prompt_messages: list[dict[str, str]]) -> str:
        text = sample.get("contract_markdown", "")
        sentences = split_sentences(text)
        candidates = regex_candidates(text)

        amount, currency = parse_amount(candidates["amounts"][0] if candidates["amounts"] else None)
        buyer_name = None
        seller_name = None
        buyer_match = re.search(r"Alıcı:\s*([^.\n]+?)(?:\s+VKN:|\.\s|$)", text)
        seller_match = re.search(r"Satıcı:\s*([^.\n]+?)(?:\s+VKN:|\.\s|$)", text)
        if buyer_match:
            buyer_name = buyer_match.group(1).strip()
        if seller_match:
            seller_name = seller_match.group(1).strip()

        tax_ids = candidates["tax_ids"]
        payment_rules = []
        for sentence in sentences:
            if "%" in sentence or "yüzde" in sentence.lower() or "ödeme" in sentence.lower() or "ödenir" in sentence.lower():
                percentages = re.findall(r"%\s*\d+(?:[.,]\d+)?|yüzde\s+\d+(?:[.,]\d+)?", sentence, flags=re.IGNORECASE)
                if percentages:
                    for pct_text in percentages:
                        pct = parse_percentage(pct_text)
                        payment_rules.append({
                            "milestone": sentence[:80],
                            "trigger": detect_trigger(sentence),
                            "percentage": pct,
                            "required_evidence": detect_evidence(sentence),
                            "source_quote": sentence,
                            "confidence": 0.65,
                        })
                elif "ödeme" in sentence.lower():
                    payment_rules.append({
                        "milestone": "Belirsiz ödeme şartı",
                        "trigger": "manual_review",
                        "percentage": None,
                        "required_evidence": detect_evidence(sentence),
                        "source_quote": sentence,
                        "confidence": 0.45,
                    })

        if not payment_rules:
            best_payment = self._best_sentence("ödeme şartı ve ödeme tetikleyicisi", sentences, ["ödeme", "ödenir"])
            payment_rules.append({
                "milestone": "Belirsiz ödeme şartı",
                "trigger": "manual_review",
                "percentage": None,
                "required_evidence": [],
                "source_quote": best_payment,
                "confidence": 0.35,
            })

        goods = []
        goods_match = re.search(r"(\d+(?:[.,]\d+)?)\s*(adet|kg|ton|paket)\s+([^.\n]+)", text, flags=re.IGNORECASE)
        if goods_match:
            goods.append({
                "name": goods_match.group(3).strip(),
                "quantity": float(goods_match.group(1).replace(",", ".")),
                "unit": goods_match.group(2),
            })

        date = candidates["dates"][0] if candidates["dates"] else None
        risk_flags = []
        if date is None:
            risk_flags.append("delivery_deadline_missing")
        if any(rule["trigger"] == "manual_review" for rule in payment_rules):
            risk_flags.append("payment_terms_unclear")

        prediction = {
            "contract_id": sample.get("id"),
            "parties": {
                "buyer": {"name": buyer_name, "tax_id": tax_ids[0] if len(tax_ids) > 0 else None},
                "seller": {"name": seller_name, "tax_id": tax_ids[1] if len(tax_ids) > 1 else None},
            },
            "commercial_terms": {
                "currency": currency,
                "total_amount": amount,
                "goods": goods,
                "delivery_deadline": date,
            },
            "payment_rules": payment_rules,
            "risk_flags": sorted(set(risk_flags)),
            "needs_manual_review": bool(risk_flags),
        }
        return json.dumps(prediction, ensure_ascii=False)

    def unload(self) -> None:
        self.model = None
        self.tokenizer = None
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

## 11. Generative Model Runner

In [13]:
class GenerativeModelRunner(BaseRunner):
    def __init__(self, model_name: str, model_id: str, precision: str = "bf16", fallback_precision: str = "4bit"):
        self.model_name = model_name
        self.model_id = model_id
        self.runner_mode = "causal_lm"
        self.runner_notes: list[str] = []
        self.precision = precision
        self.fallback_precision = fallback_precision
        self.tokenizer = None
        self.model = None
        self.loaded_precision = None
        self._load_with_fallback()

    def _load_with_fallback(self) -> None:
        try:
            self._load(self.precision)
            self.loaded_precision = self.precision
        except Exception as exc:
            note = f"{self.model_name} {self.precision} load failed: {exc}"
            print(note)
            self.runner_notes.append(note)
            if self.precision == self.fallback_precision:
                raise
            self.unload()
            print(f"Retrying {self.model_name} with {self.fallback_precision}")
            self._load(self.fallback_precision)
            self.loaded_precision = self.fallback_precision

    def _load(self, precision: str) -> None:
        if torch is None:
            raise RuntimeError("torch is required for generative model loading")
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id, trust_remote_code=True, token=HF_TOKEN)
        kwargs = {
            "device_map": "auto",
            "trust_remote_code": True,
            "token": HF_TOKEN,
        }
        if precision == "4bit":
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
        elif precision == "bf16":
            kwargs["torch_dtype"] = torch.bfloat16
        elif precision == "fp16":
            kwargs["torch_dtype"] = torch.float16
        else:
            raise ValueError(f"Unknown precision: {precision}")

        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, **kwargs)
        self.model.eval()
        print(f"Loaded {self.model_name} with {precision}")

    def _format_prompt(self, prompt_messages: list[dict[str, str]]) -> str:
        try:
            return self.tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            system = prompt_messages[0]["content"]
            user = prompt_messages[1]["content"]
            return f"SYSTEM:\n{system}\n\nUSER:\n{user}\n\nASSISTANT:\n"

    def run(self, sample: dict[str, Any], prompt_messages: list[dict[str, str]]) -> str:
        if torch is None or self.tokenizer is None or self.model is None:
            raise RuntimeError("Model is not loaded")
        prompt = self._format_prompt(prompt_messages)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, **GENERATION_CONFIG)
        new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    def unload(self) -> None:
        self.model = None
        self.tokenizer = None
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()

## 12. Benchmark Runner

In [14]:
def get_peak_vram_gb() -> float:
    if torch is None or not torch.cuda.is_available():
        return 0.0
    return round(torch.cuda.max_memory_allocated() / 1024**3, 3)


def load_runner(model_name: str) -> BaseRunner:
    config = MODEL_REGISTRY[model_name]
    if config.get("requires_hf_token") and not HF_TOKEN:
        raise RuntimeError(f"{model_name} requires HF_TOKEN but HF_TOKEN is not set")
    if config["kind"] == "encoder_baseline":
        return BerturkBaselineRunner(model_name, config["model_id"])
    precision = config.get("default_precision", DEFAULT_GENERATIVE_PRECISION)
    if RUN_PROFILE != "h100_full":
        precision = "4bit"
    return GenerativeModelRunner(
        model_name=model_name,
        model_id=config["model_id"],
        precision=precision,
        fallback_precision=config.get("fallback_precision", "4bit"),
    )


def save_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def select_samples_for_model(model_name: str, samples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    config = MODEL_REGISTRY[model_name]
    if SMOKE_14B_SINGLE_SAMPLE and config["kind"] == "causal_lm" and "14b" in model_name.lower():
        print(f"{model_name}: 14B smoke mode active, running only 1 sample.")
        return samples[:1]
    return samples


def build_result_row(
    model_name: str,
    runner_mode: str,
    sample: dict[str, Any],
    raw_output: str,
    parsed: dict[str, Any],
    validation: dict[str, Any],
    metrics: dict[str, float],
    latency_sec: float,
    runner_notes: list[str] | None = None,
) -> dict[str, Any]:
    return {
        "model_name": model_name,
        "model_kind": MODEL_REGISTRY[model_name]["kind"],
        "model_id": MODEL_REGISTRY[model_name]["model_id"],
        "runner_mode": runner_mode,
        "runner_notes": runner_notes or [],
        "sample_id": sample["id"],
        "raw_output": raw_output,
        "parsed_json": parsed["parsed_json"],
        "parse_success": parsed["parse_success"],
        "parse_error": parsed["parse_error"],
        "latency_sec": round(latency_sec, 3),
        "peak_vram_gb": get_peak_vram_gb(),
        **validation,
        **metrics,
    }


def run_benchmark(samples: list[dict[str, Any]], model_names: list[str]) -> list[dict[str, Any]]:
    all_results: list[dict[str, Any]] = []
    for model_name in model_names:
        if not RUN_MODELS.get(model_name, False):
            continue
        print(f"\n=== Running {model_name} ===")
        model_results = []
        runner = None
        try:
            runner = load_runner(model_name)
            if torch is not None and torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            model_samples = select_samples_for_model(model_name, samples)
            for sample in tqdm(model_samples, desc=model_name):
                prompt_messages = build_prompt(sample)
                started = time.time()
                try:
                    raw_output = runner.run(sample, prompt_messages)
                    latency_sec = time.time() - started
                    parsed = parse_model_json(raw_output)
                    validation = validate_prediction(parsed["parsed_json"], sample)
                    metrics = score_prediction(parsed["parsed_json"], sample["gold"], sample)
                    result = build_result_row(
                        model_name=model_name,
                        runner_mode=getattr(runner, "runner_mode", MODEL_REGISTRY[model_name]["kind"]),
                        sample=sample,
                        raw_output=raw_output,
                        parsed=parsed,
                        validation=validation,
                        metrics=metrics,
                        latency_sec=latency_sec,
                        runner_notes=getattr(runner, "runner_notes", []),
                    )
                except Exception as sample_exc:
                    result = build_result_row(
                        model_name=model_name,
                        runner_mode=getattr(runner, "runner_mode", MODEL_REGISTRY[model_name]["kind"]),
                        sample=sample,
                        raw_output="",
                        parsed={"parse_success": False, "parsed_json": None, "parse_error": str(sample_exc)},
                        validation={
                            "schema_valid": False,
                            "validator_pass": False,
                            "validator_errors": [f"runtime_error: {sample_exc}"],
                        },
                        metrics=score_prediction(None, sample["gold"], sample),
                        latency_sec=time.time() - started,
                        runner_notes=getattr(runner, "runner_notes", []),
                    )
                model_results.append(result)
                all_results.append(result)
        except Exception as model_exc:
            print(f"Model skipped/failed: {model_name}: {model_exc}")
            skip_rows = []
            for sample in select_samples_for_model(model_name, samples):
                row = build_result_row(
                    model_name=model_name,
                    runner_mode="causal_lm" if MODEL_REGISTRY[model_name]["kind"] == "causal_lm" else MODEL_REGISTRY[model_name]["kind"],
                    sample=sample,
                    raw_output="",
                    parsed={"parse_success": False, "parsed_json": None, "parse_error": str(model_exc)},
                    validation={
                        "schema_valid": False,
                        "validator_pass": False,
                        "validator_errors": [f"model_skipped: {model_exc}"],
                    },
                    metrics=score_prediction(None, sample["gold"], sample),
                    latency_sec=0.0,
                    runner_notes=[f"Model skipped/failed: {model_exc}"],
                )
                skip_rows.append(row)
                all_results.append(row)
            model_results = skip_rows
        finally:
            if runner is not None:
                runner.unload()
        save_jsonl(RAW_DIR / f"{model_name}_predictions.jsonl", model_results)
        print(f"Saved {len(model_results)} rows for {model_name}")
    return all_results


enabled_models = [name for name, enabled in RUN_MODELS.items() if enabled]
RESULTS = run_benchmark(SAMPLES, enabled_models)
len(RESULTS)


=== Running berturk ===


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

...

Loaded deepseek_r1_qwen_14b with bf16


deepseek_r1_qwen_14b:   0%|          | 0/20 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_toke

Saved 20 rows for deepseek_r1_qwen_14b

=== Running llama31_8b ===
llama31_8b bf16 load failed: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-6a50b6c5-601778795c8cfabf76aa0fa0;7a07c868-dd22-4a64-ae71-3fdb3a039e71)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct to ask for access.
Retrying llama31_8b with 4bit
Model skipped/failed: llama31_8b: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-6a50b6c5-4ccf605e4ef656141b495f6b;6ba75781-9b06-4ce6-94cd-403d73363dde)

Cannot access gated repo for url https://huggingface.co/met

80

## 13. Results Table

In [15]:
def summarize_results(results: list[dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    per_sample = pd.DataFrame(results)
    if per_sample.empty:
        return per_sample, pd.DataFrame()
    metric_columns = [
        "parse_success", "schema_valid", "validator_pass", "latency_sec", "peak_vram_gb",
        "party_name_fuzzy_f1", "tax_id_exact_accuracy", "currency_accuracy",
        "total_amount_accuracy", "delivery_deadline_accuracy", "goods_name_fuzzy_f1",
        "goods_quantity_accuracy", "payment_rule_count_accuracy", "payment_percentage_accuracy",
        "trigger_accuracy", "required_evidence_f1", "risk_flags_f1",
        "needs_manual_review_accuracy", "source_quote_grounding_rate", "critical_field_pass_rate", "field_f1",
    ]
    agg = {column: "mean" for column in metric_columns if column in per_sample.columns}
    group_columns = ["model_name", "model_kind", "model_id"]
    if "runner_mode" in per_sample.columns:
        group_columns.append("runner_mode")
    summary = per_sample.groupby(group_columns, dropna=False).agg(agg).reset_index()
    return per_sample, summary


PER_SAMPLE_DF, SUMMARY_DF = summarize_results(RESULTS)
display(SUMMARY_DF)

,model_name,model_kind,model_id,runner_mode,parse_success,schema_valid,validator_pass,latency_sec,peak_vram_gb,party_name_fuzzy_f1,...,goods_quantity_accuracy,payment_rule_count_accuracy,payment_percentage_accuracy,trigger_accuracy,required_evidence_f1,risk_flags_f1,needs_manual_review_accuracy,source_quote_grounding_rate,critical_field_pass_rate,field_f1
0,berturk,encoder_baseline,dbmdz/bert-base-turkish-cased,berturk_encoder,1.0,1.0,0.00,0.03675,0.41390,0.276405,...,0.9,0.60,0.40,0.350000,0.383333,0.0,0.35,0.912500,0.468750,0.537303
1,deepseek_r1_qwen_14b,causal_lm,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,causal_lm,1.0,0.8,0.30,45.67545,27.83680,1.000000,...,0.7,1.00,0.75,0.403333,0.516667,0.4,0.65,0.975000,0.850417,0.756786
2,llama31_8b,causal_lm,meta-llama/Llama-3.1-8B-Instruct,causal_lm,0.0,0.0,0.00,0.00000,27.84400,0.000000,...,0.0,0.00,0.00,0.000000,0.000000,0.0,0.00,0.000000,0.000000,0.000000
3,qwen3_14b,causal_lm,Qwen/Qwen3-14B,causal_lm,0.8,0.7,0.55,59.83240,27.85635,0.750000,...,0.7,0.75,0.65,0.320000,0.405952,0.3,0.65,0.733333,0.690000,0.636378


## 13.1 Benchmark Run Warnings

In [16]:
def benchmark_warning_reasons(results: list[dict[str, Any]], samples: list[dict[str, Any]]) -> list[str]:
    reasons = []
    if results and any(row.get("runner_mode") == "regex_only_fallback" for row in results):
        reasons.append("regex_only_fallback runner was used")
    if len(samples) < 20:
        reasons.append("dataset size is below 20")
    generative_model_names = [name for name, cfg in MODEL_REGISTRY.items() if cfg["kind"] == "causal_lm"]
    generative_enabled = [name for name in generative_model_names if RUN_MODELS.get(name, False)]
    if not generative_enabled:
        reasons.append("generative models are all disabled")
    if results:
        model_kinds = {row.get("model_kind") for row in results}
        runner_modes = {row.get("runner_mode") for row in results}
        if model_kinds <= {"encoder_baseline"} or runner_modes <= {"berturk_encoder", "regex_only_fallback"}:
            reasons.append("only BERTurk/regex baseline was run")
    return reasons


WARNING_REASONS = benchmark_warning_reasons(RESULTS, SAMPLES)
if WARNING_REASONS:
    print("This is not a full open-source LLM benchmark yet; it is a smoke/baseline run.")
    print("Reasons:")
    for reason in WARNING_REASONS:
        print("-", reason)

## 14. Error Analysis

In [17]:
def worst_examples(per_sample: pd.DataFrame, n: int = 3) -> dict[str, pd.DataFrame]:
    output = {}
    if per_sample.empty:
        return output
    for model_name, group in per_sample.groupby("model_name"):
        sort_cols = ["field_f1", "parse_success", "validator_pass"]
        existing = [col for col in sort_cols if col in group.columns]
        output[model_name] = group.sort_values(existing, ascending=True).head(n)
    return output


WORST = worst_examples(PER_SAMPLE_DF)
for model_name, frame in WORST.items():
    print(f"\n### {model_name} worst examples")
    display(frame[["sample_id", "field_f1", "parse_success", "schema_valid", "validator_pass", "parse_error", "validator_errors"]])


### berturk worst examples


,sample_id,field_f1,parse_success,schema_valid,validator_pass,parse_error,validator_errors
0,M4T-001,0.226190,True,True,False,None,"[payment_rules[0].percentage_missing, payment_..."
2,M4T-003,0.357143,True,True,False,None,"[payment_rules[0].percentage_missing, payment_..."
19,M4T-020,0.390394,True,True,False,None,"[payment_rules[0].percentage_missing, payment_..."



### deepseek_r1_qwen_14b worst examples


,sample_id,field_f1,parse_success,schema_valid,validator_pass,parse_error,validator_errors
50,M4T-011,0.571429,True,False,False,None,"[schema_error: [{'type': 'list_type', 'loc': (..."
51,M4T-012,0.571429,True,False,False,None,"[schema_error: [{'type': 'list_type', 'loc': (..."
59,M4T-020,0.619048,True,True,True,None,[]



### llama31_8b worst examples


,sample_id,field_f1,parse_success,schema_valid,validator_pass,parse_error,validator_errors
60,M4T-001,0.0,False,False,False,You are trying to access a gated repo.\nMake s...,[model_skipped: You are trying to access a gat...
61,M4T-002,0.0,False,False,False,You are trying to access a gated repo.\nMake s...,[model_skipped: You are trying to access a gat...
62,M4T-003,0.0,False,False,False,You are trying to access a gated repo.\nMake s...,[model_skipped: You are trying to access a gat...



### qwen3_14b worst examples


,sample_id,field_f1,parse_success,schema_valid,validator_pass,parse_error,validator_errors
20,M4T-001,0.0,False,False,False,"Expecting ',' delimiter: line 24 column 4 (cha...",[prediction_missing]
27,M4T-008,0.0,False,False,False,"Expecting ',' delimiter: line 41 column 6 (cha...",[prediction_missing]
37,M4T-018,0.0,False,False,False,"Expecting ',' delimiter: line 24 column 4 (cha...",[prediction_missing]


## 15. Final Recommendation

In [18]:
def classify_model(row: pd.Series) -> str:
    if len(SAMPLES) < 10:
        return "smoke only"
    if row.get("runner_mode") == "regex_only_fallback":
        return "baseline only / not an LLM replacement"
    if (
        row.get("parse_success", 0) >= 0.95
        and row.get("validator_pass", 0) >= 0.8
        and row.get("field_f1", 0) >= 0.8
        and row.get("critical_field_pass_rate", 0) >= 0.8
    ):
        return "mümkün ve hafif"
    if row.get("field_f1", 0) >= 0.6 and row.get("critical_field_pass_rate", 0) < 0.8:
        return "mümkün ama veri/kaynak lazım"
    if row.get("parse_success", 0) >= 0.7 and row.get("field_f1", 0) >= 0.6:
        return "mümkün ama veri/kaynak lazım"
    return "MVP için hosted LLM daha mantıklı"


if not SUMMARY_DF.empty:
    SUMMARY_DF = SUMMARY_DF.copy()
    SUMMARY_DF["final_recommendation"] = SUMMARY_DF.apply(classify_model, axis=1)
    display_cols = ["model_name", "model_kind", "runner_mode", "parse_success", "validator_pass", "field_f1", "critical_field_pass_rate", "latency_sec", "peak_vram_gb", "final_recommendation"]
    display(SUMMARY_DF[[col for col in display_cols if col in SUMMARY_DF.columns]])
else:
    print("No benchmark results yet.")

,model_name,model_kind,runner_mode,parse_success,validator_pass,field_f1,critical_field_pass_rate,latency_sec,peak_vram_gb,final_recommendation
0,berturk,encoder_baseline,berturk_encoder,1.0,0.00,0.537303,0.468750,0.03675,0.41390,MVP için hosted LLM daha mantıklı
1,deepseek_r1_qwen_14b,causal_lm,causal_lm,1.0,0.30,0.756786,0.850417,45.67545,27.83680,mümkün ama veri/kaynak lazım
2,llama31_8b,causal_lm,causal_lm,0.0,0.00,0.000000,0.000000,0.00000,27.84400,MVP için hosted LLM daha mantıklı
3,qwen3_14b,causal_lm,causal_lm,0.8,0.55,0.636378,0.690000,59.83240,27.85635,mümkün ama veri/kaynak lazım


## 16. Export Artifacts

In [19]:
def export_artifacts(results: list[dict[str, Any]], output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    metrics_dir = output_dir / "metrics"
    reports_dir = output_dir / "reports"
    metrics_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)

    per_sample, summary = summarize_results(results)
    per_sample_csv = metrics_dir / "per_sample_metrics.csv"
    summary_csv = metrics_dir / "summary_metrics.csv"
    per_sample.to_csv(per_sample_csv, index=False)
    summary.to_csv(summary_csv, index=False)

    def dataframe_to_markdown(frame: pd.DataFrame) -> str:
        if frame.empty:
            return ""
        columns = list(frame.columns)
        header = "| " + " | ".join(columns) + " |"
        separator = "| " + " | ".join(["---"] * len(columns)) + " |"
        rows = []
        for _, row in frame.iterrows():
            values = []
            for column in columns:
                value = row[column]
                if isinstance(value, float):
                    value = f"{value:.3f}"
                values.append(str(value).replace("\n", " "))
            rows.append("| " + " | ".join(values) + " |")
        return "\n".join([header, separator, *rows])

    lines = [
        "# M4Trust Open-Source Model Benchmark Report",
        "",
        "## Setup",
        "",
        f"- Runtime profile: {RUN_PROFILE}",
        f"- GPU: {GPU_INFO.get('gpu_name')}",
        f"- VRAM GB: {GPU_INFO.get('vram_gb')}",
        f"- Dataset size: {len(SAMPLES)}",
        f"- Models tested: {', '.join(sorted(set(row['model_name'] for row in results))) if results else 'none'}",
        f"- Model run mode hint: {DATASET_REPORT.get('model_run_mode_hint', 'unknown') if 'DATASET_REPORT' in globals() else 'unknown'}",
        "",
        "## Dataset Report",
        "",
        f"- Dataset size: {DATASET_REPORT.get('dataset_size', len(SAMPLES)) if 'DATASET_REPORT' in globals() else len(SAMPLES)}",
        f"- Manual review count: {DATASET_REPORT.get('manual_review_count', 'unknown') if 'DATASET_REPORT' in globals() else 'unknown'}",
        f"- Risk sample count: {DATASET_REPORT.get('risk_sample_count', 'unknown') if 'DATASET_REPORT' in globals() else 'unknown'}",
        f"- Trigger distribution: {json.dumps(DATASET_REPORT.get('trigger_distribution', {}), ensure_ascii=False) if 'DATASET_REPORT' in globals() else '{}'}",
        f"- Payment rule count distribution: {json.dumps(DATASET_REPORT.get('payment_rule_count_distribution', {}), ensure_ascii=False) if 'DATASET_REPORT' in globals() else '{}'}",
        "",
        "## Warning Reasons",
        "",
        "Warning reasons:",
        "",
        *(f"- {reason}" for reason in (WARNING_REASONS if 'WARNING_REASONS' in globals() else benchmark_warning_reasons(results, SAMPLES))),
        "",
        "## Summary Table",
        "",
    ]
    if summary.empty:
        lines.append("No benchmark results were produced.")
    else:
        summary_for_report = summary.copy()
        summary_for_report["final_recommendation"] = summary_for_report.apply(classify_model, axis=1)
        report_cols = [
            "model_name", "model_kind", "runner_mode", "parse_success", "schema_valid", "field_f1", "critical_field_pass_rate",
            "validator_pass", "latency_sec", "peak_vram_gb", "final_recommendation",
        ]
        lines.append(dataframe_to_markdown(summary_for_report[[col for col in report_cols if col in summary_for_report.columns]]))
    lines.extend(["", "## Per-model Notes", ""])
    if results:
        per_sample = pd.DataFrame(results)
        for model_name, group in per_sample.groupby("model_name"):
            kind = group["model_kind"].iloc[0]
            lines.extend([
                f"### {model_name}",
                "",
                f"- Kind: {kind}",
                f"- Runner modes: {', '.join(sorted(str(value) for value in group['runner_mode'].dropna().unique())) if 'runner_mode' in group else 'unknown'}",
                f"- Parse success: {group['parse_success'].mean():.3f}",
                f"- Validator pass: {group['validator_pass'].mean():.3f}",
                f"- Field F1: {group['field_f1'].mean():.3f}",
                f"- Critical field pass rate: {group['critical_field_pass_rate'].mean():.3f}",
                f"- Avg latency sec: {group['latency_sec'].mean():.3f}",
                f"- Peak VRAM GB: {group['peak_vram_gb'].max():.3f}",
                "",
            ])
            if kind == "encoder_baseline":
                lines.append("BERTurk is evaluated as an encoder-based extraction baseline, not as a generative JSON model.")
                notes = []
                if "runner_notes" in group:
                    for value in group["runner_notes"]:
                        if isinstance(value, list):
                            notes.extend(str(item) for item in value)
                        elif value:
                            notes.append(str(value))
                unique_notes = sorted(set(notes))
                if unique_notes:
                    lines.append("BERTurk runner notes:")
                    for note in unique_notes:
                        lines.append(f"- {note}")
                lines.append("")
            worst = group.sort_values(["field_f1", "parse_success", "validator_pass"], ascending=True).head(3)
            lines.append("Worst examples:")
            for _, row in worst.iterrows():
                lines.append(f"- {row['sample_id']}: field_f1={row['field_f1']:.3f}, parse_success={row['parse_success']}, errors={row['validator_errors']}")
            lines.append("")
    lines.extend([
        "## Final Recommendation",
        "",
        "M4Trust demo akışında hosted LLM kullanır. Bu notebook açık kaynak modellerin aynı extraction görevindeki fizibilitesini ölçen ar-ge kanıtıdır.",
        "",
        "Possible classes:",
        "",
        "- mümkün ve hafif",
        "- mümkün ama veri/kaynak lazım",
        "- MVP için hosted LLM daha mantıklı",
        "- baseline only / not an LLM replacement",
        "- smoke only",
        "",
    ])
    report_path = reports_dir / "benchmark_report.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print("Exported:")
    print("-", per_sample_csv)
    print("-", summary_csv)
    print("-", report_path)


export_artifacts(RESULTS, OUTPUT_DIR)

Exported:
- /content/drive/MyDrive/m4trust_benchmark/outputs/metrics/per_sample_metrics.csv
- /content/drive/MyDrive/m4trust_benchmark/outputs/metrics/summary_metrics.csv
- /content/drive/MyDrive/m4trust_benchmark/outputs/reports/benchmark_report.md
